# Create Work Citations (oxjob #2314, shadow)

One row per citation: the citing work, the cited work, and every input that holds it (each filed reference list with its
version tier, legacy MAG, or the frozen per-location arrays). Built once per night after `parse_referenced_works` and
`Merged_Work_Ids_Map`.

**An exact copy of what `Works_Enriched` serves today** (Casey, 17 Nov 2026: move to one table first, change no
citation). `served` is true on every row. Rule changes come later, each its own decision, as changes to `served`.

**Shadow:** nothing reads `openalex.works.work_citations` yet and an error here only logs. The switch makes
`Works_Enriched` set each work's `referenced_works` from the served rows (sorted, first 5,000).
Rehearsed on 17 Nov's tables: 3,594,657,945 rows; every one of the 3,593,263,540 published citations is in it; the extra
1,394,405 are on works over the 5,000 cap. 4 min on shared-2xl.

In [0]:
-- oxjob #2314, SHADOW: one row per citation (citing work, cited work) with every input that holds it, built once per night.
-- It holds exactly what CreateWorksEnriched serves today (Casey 17 Nov: move to one table first, change no citation): our
-- filed lists (work_references, each list with its version tier: published and accepted 1, submitted 2, other 3), legacy
-- MAG (openalex.mid.citation) and the frozen per-location arrays (openalex_works_base.referenced_works, from
-- location_enrichments). Both ends follow tonight's merged_work_ids_map; no work cites itself; the citing work is one of
-- tonight's works. served = every row, as today. Rule changes (MAG only where a work's lists link nothing, rule B, the
-- frozen arrays retired) come later, one night each, as changes to served. Nothing reads this table yet; any error only logs.
BEGIN
  DECLARE EXIT HANDLER FOR SQLEXCEPTION
  BEGIN
    DECLARE msg STRING;
    GET DIAGNOSTICS CONDITION 1 msg = MESSAGE_TEXT;
    SELECT concat('work_citations (shadow, oxjob #2314): build failed, nothing reads it yet so End 2 End goes on: ', msg) AS work_citations;
  END;
  CREATE OR REPLACE TABLE openalex.works.work_citations
  CLUSTER BY (citing_work_id)
  TBLPROPERTIES ('delta.deletedFileRetentionDuration' = '7 days', 'delta.logRetentionDuration' = '7 days')
  AS
  WITH m AS (SELECT id AS k, merge_into_id AS v FROM openalex.works.merged_work_ids_map),
  lists AS (
    SELECT l.native_id, l.native_id_namespace, l.provenance, coalesce(mc.v, l.citing_work_id) AS citing_work_id,
      CASE WHEN l.version_rank <= 2 THEN 1 WHEN l.version_rank = 3 THEN 2 ELSE 3 END AS version_tier
    FROM openalex.works.work_reference_lists_current l
    LEFT JOIN m mc ON mc.k = l.citing_work_id
  ),
  best AS (SELECT citing_work_id, min(version_tier) AS best_tier FROM lists GROUP BY citing_work_id),
  edges AS (
    SELECT coalesce(mc.v, wr.citing_work_id) AS citing_work_id, coalesce(md.v, wr.cited_work_id) AS cited_work_id,
      named_struct('provenance', wr.provenance, 'native_id_namespace', wr.native_id_namespace, 'native_id', wr.native_id,
                   'version_tier', l.version_tier) AS source
    FROM openalex.works.work_references wr
    LEFT JOIN lists l
      ON l.native_id = wr.native_id AND l.native_id_namespace = wr.native_id_namespace AND l.provenance = wr.provenance
    LEFT JOIN m mc ON mc.k = wr.citing_work_id
    LEFT JOIN m md ON md.k = wr.cited_work_id
    WHERE wr.cited_work_id IS NOT NULL
    UNION ALL
    SELECT coalesce(mc.v, c.paper_id), coalesce(md.v, c.paper_reference_id),
      named_struct('provenance', 'legacy-mag', 'native_id_namespace', CAST(NULL AS STRING), 'native_id', CAST(NULL AS STRING),
                   'version_tier', CAST(NULL AS INT))
    FROM openalex.mid.citation c
    LEFT JOIN m mc ON mc.k = c.paper_id
    LEFT JOIN m md ON md.k = c.paper_reference_id
    UNION ALL
    SELECT x.id, coalesce(md.v, x.cited),
      named_struct('provenance', 'frozen-array', 'native_id_namespace', CAST(NULL AS STRING), 'native_id', CAST(NULL AS STRING),
                   'version_tier', CAST(NULL AS INT))
    FROM (SELECT id, explode(referenced_works) AS cited FROM openalex.works.openalex_works_base WHERE size(referenced_works) > 0) x
    LEFT JOIN m md ON md.k = x.cited
  ),
  pairs AS (
    SELECT citing_work_id, cited_work_id, array_sort(collect_set(source)) AS sources
    FROM edges
    WHERE citing_work_id IS NOT NULL AND cited_work_id IS NOT NULL AND citing_work_id <> cited_work_id
    GROUP BY citing_work_id, cited_work_id
  )
  SELECT p.citing_work_id, p.cited_work_id, p.sources,
    coalesce(exists(p.sources, s -> s.version_tier = b.best_tier), FALSE) AS in_best_version,
    exists(p.sources, s -> s.provenance = 'legacy-mag') AS from_legacy_mag,
    exists(p.sources, s -> s.provenance = 'frozen-array') AS from_frozen_arrays,
    TRUE AS served,
    current_timestamp() AS built_at
  FROM pairs p
  LEFT JOIN best b ON b.citing_work_id = p.citing_work_id
  LEFT SEMI JOIN openalex.works.openalex_works_base w ON w.id = p.citing_work_id;
END
